# 6. 샌드박스

코드 실행 도구는 작업별 도구를 따로 만들 필요를 줄이지만 되돌릴 수 없는 작업도 허용한다.
실행 경계를 단계별로 구성하고 차단 범위를 확인한다.
</br>범용으로 사용할만한 도구를 하나 제시

### 공통 설정

`1`과 같은 블록이다. 프로세스 실행에 `subprocess`를 쓴다.

In [1]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []

def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

모델이 쓴 코드가 도는 작업 디렉터리 `box/`를 비우고 만든다. 절대 경로로 잡아 두어야 뒤의 경로 검사가 된다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph").resolve()
BOX = Path("box").resolve()
shutil.rmtree(BOX, ignore_errors=True)
BOX.mkdir()
print(BOX)

/home/ubuntu/work/kosa-ict-genai-2026-hyundai/src/exercise/csmoon1010/day08/lecture/box


### exec로 하면 안 되는 이유

같은 프로세스에서 돌리면 모델이 쓴 코드가 우리 변수와 같은 공간에 있다.

In [3]:
CORE_LIMIT = 40
model_code = "CORE_LIMIT = 0"  # 모델이 쓴 코드라고 하자
exec(model_code)
print("CORE_LIMIT =", CORE_LIMIT)
CORE_LIMIT = 40

CORE_LIMIT = 0


우리 상태가 덮였다. 무한 루프였다면 커널을 죽여야 했을 것이다. `exec`는 돌기 시작하면 밖에서 끊을 방법이 없어 타임아웃도 걸 수 없다. => 별도의 프로세스로 진행

### 프로세스로 가른다

별도 프로세스로 보내면 세 가지가 생긴다. 상태 격리, 타임아웃, 출력 크기 제한.
작업 디렉터리를 `box/`로 고정해서 상대 경로로 만든 파일이 그 안에만 생기게 한다.

In [4]:
def run_python(code, seconds=5, cap=2000):  # 5초안에 실행, 출력길이 2000자 제한
    try:
        p = subprocess.run([sys.executable, "-c", code], cwd=BOX, timeout=seconds,
                           capture_output=True, text=True)
    except subprocess.TimeoutExpired:
        return f"error: {seconds}초를 넘겨 중단했다. 더 작은 범위로 나눠 다시 실행한다."
    out = (p.stdout + p.stderr).strip()
    if len(out) > cap:
        out = out[:cap] + f"\n... [잘림. 전체 {len(out):,}자 중 앞 {cap:,}자]"
    return out or "(출력 없음)"


print(run_python("print(sum(range(10)))"))

45


세 가지 실패를 넣어 본다. 모두 문자열로 돌아오고 커널은 살아 있다.

In [5]:
for label, code in [("타임아웃", "import time; time.sleep(30)"),
                    ("출력 폭주", "print('가' * 100000)"),
                    ("예외", "1 / 0")]:
    print(f"{label:6} {run_python(code)[:88]}")

타임아웃   error: 5초를 넘겨 중단했다. 더 작은 범위로 나눠 다시 실행한다.
출력 폭주  가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가
예외     Traceback (most recent call last):
  File "<string>", line 1, in <module>
ZeroDivisionEr


### 파일 도구는 스코프를 검사한다

`cwd`는 상대 경로만 잡아 준다. 절대 경로나 `..`로는 밖으로 나간다.
그래서 파일 도구 쪽에서 경로를 검사한다. `2`의 인자 검증과 같은 자리다.

In [6]:
def inside(path, root):
    target = (root / path).resolve()
    return target if target == root or root in target.parents else None


def write_file(path, text): # path 경로에 text를 써라
    target = inside(path, BOX)  # BOX 경로 안이 맞는지 체크
    if target is None:
        raise ValueError(f"'{path}' 는 box/ 밖이다")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")
    return f"{path} 에 {len(text)}자"

In [7]:
def read_file(path):
    target = inside(path, BOX)
    if target is None:
        raise ValueError(f"'{path}' 는 box/ 밖이다")
    return target.read_text(encoding="utf-8")[:2000]


def list_files():
    return sorted(str(f.relative_to(BOX)) for f in BOX.rglob("*") if f.is_file())

안쪽 경로 하나와 밖으로 나가는 경로 둘을 넣어 본다.

In [8]:
print(write_file("note.txt", "첫 줄"))
for bad in ["../PLAN.md", "/etc/hostname"]:
    try:
        read_file(bad)
    except ValueError as e:
        print("거부", e)

note.txt 에 3자
거부 '../PLAN.md' 는 box/ 밖이다
거부 '/etc/hostname' 는 box/ 밖이다


### 격리는 어디까지인가

파일 도구는 막았다. 그런데 코드 실행 도구는 파이썬 전체를 준 것이다.
`open()`을 우리가 검사할 수 없다.

In [9]:
print(run_python(f"print(open({str(CORPUS.parent.parent.parent / 'README.md')!r}, encoding='utf-8').read()[:60])"))

# kosa-ict-genai-2026-hyundai


`cwd`는 상대 경로만 잡는다. 프로세스가 열 수 있는 것은 이 계정이 열 수 있는 전부다.

### 컨테이너 격리

프로세스 격리로 막지 못하는 접근은 커널 경계에서 차단한다. 네트워크를 끊고 메모리와 CPU를 제한하며 `box/`만 마운트한다.
</br>
도커 설치 : https://get.docker.com/

In [10]:
DOCKER = ["docker", "run", "--rm", "--network", "none", "-m", "256m", "--cpus", "1",
          "-v", f"{BOX}:/box", "-w", "/box", "python:3.11-slim", "python", "-c"]
print(" ".join(DOCKER) + " '<code>'")


def run_docker(code, seconds=30):
    try:
        p = subprocess.run(DOCKER + [code], timeout=seconds, capture_output=True, text=True)
        return (p.stdout + p.stderr).strip().splitlines()[-1][:70]
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        return f"건너뜀: {type(e).__name__}"

docker run --rm --network none -m 256m --cpus 1 -v /home/ubuntu/work/kosa-ict-genai-2026-hyundai/src/exercise/csmoon1010/day08/lecture/box:/box -w /box python:3.11-slim python -c '<code>'


같은 코드를 프로세스와 컨테이너에서 돌려 비교한다. 도커가 없으면 건너뛰며 격리는 프로세스 경계까지만 남는다.

In [12]:
outside = str(CORPUS.parent.parent.parent / "README.md")
if shutil.which("docker"):
    print("프로세스에서 바깥 읽기 |", run_python(f"print(open({outside!r}).read()[:20])")[:40])
    print("컨테이너에서 같은 것  |", run_docker(f"print(open({outside!r}).read()[:20])"))
    print("컨테이너에서 네트워크 |", run_docker("import urllib.request as u; u.urlopen('https://example.com')"))
    print("컨테이너에서 box/    |", run_docker("import os; print(sorted(os.listdir('/box')))"))
else:
    print("도커가 없다. 이 셀은 건너뛴다.")

프로세스에서 바깥 읽기 | # kosa-ict-genai-202
컨테이너에서 같은 것  | FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/
컨테이너에서 네트워크 | urllib.error.URLError: <urlopen error [Errno -3] Temporary failure in 
컨테이너에서 box/    | ['note.txt']


### 코어에 붙인다

`5`의 코어 블록을 그대로 복제하고 `TOOLS`와 `IMPL`과 `CONFIG`만 이 회차의 것으로 바꾼다.

In [13]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("run_python", f"파이썬 코드를 실행하고 표준 출력을 돌려준다."
                            f" 작업 디렉터리는 box/ 이고 코퍼스는 {CORPUS} 에 있다.",
              {"code": {"type": "string", "description": "실행할 코드"}}),
         spec("list_files", "box/ 안의 파일 목록을 돌려준다.", {}),
         spec("read_file", "box/ 안의 파일을 읽는다.",
              {"path": {"type": "string", "description": "box/ 기준 상대 경로"}})]

IMPL = {"run_python": run_python, "list_files": list_files, "read_file": read_file}

코드 실행은 부작용이 있지만 여기서는 게이트를 비워 둔다. 승인 없이 돌아가는 경계가 어디까지인지 보는 것이 이 회차다.

In [14]:
CONFIG = {
    "instructions": "코드를 실행해 확인한 것만 답한다. 결과 파일은 현재 디렉터리에 만든다.",
    "max_turns": 8, "max_tokens": 60000, "max_seconds": 180,
    "gated": [],
    "gate_error": "error: 이 도구는 승인이 필요하다. 무엇을 할지 사용자에게 먼저 확인한다.",
    "tool_error": "error: {e}. 인자를 고쳐 다시 부른다.",
}

In [15]:
def call_tool(name, args, approve):
    if name in CONFIG["gated"] and not approve:
        return CONFIG["gate_error"]
    try:
        return json.dumps(IMPL[name](**args), ensure_ascii=False)
    except Exception as e:
        return CONFIG["tool_error"].format(e=e)


def run(task, approve=False):
    messages = [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > CONFIG["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > CONFIG["max_seconds"]:
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), approve)}
                     for c in calls]
    return None, "반복 예산", trace

### 도구 하나로 하던 일을 코드가 대신한다

`5`까지는 `grep_docs`라는 도구를 우리가 만들어 주었다.
아래 과제는 그 도구가 없다. 세는 방법을 모델이 코드로 써야 한다.

In [16]:
answer, why, trace = run("코퍼스에서 글자 수가 가장 많은 문서 세 편의 제목과 글자 수를 센다."
                         " 결과를 top3.json 에도 저장한다")
print(why, "|", answer[:260] if answer else answer)

완료 | 확인 결과 상위 3편은 다음과 같습니다.

1. 이승만 — 138028자
2. 서재필 — 98758자
3. 김구 — 91490자

또한 결과를 `top3.json`에 저장했습니다.


턴마다 무엇을 돌렸는지와 `box/`에 남은 파일을 본다. llm이 만든 python 코드를 실행한 것!

In [17]:
for t in trace:
    print(f"  턴 {t['턴']}  {t['초']:5.1f}초  {t['도구']}")
print("남은 파일", list_files())

  턴 1    6.0초  ['run_python']
  턴 2    9.0초  ['run_python']
  턴 3   12.3초  ['run_python']
  턴 4   14.1초  []
남은 파일 ['note.txt', 'top3.json']


`*.md`만 센 값과 대조한다. 코퍼스 디렉터리에는 문서 외의 파일도 있다.

In [18]:
truth = sorted(((len(f.read_text(encoding="utf-8")), f.stem) for f in CORPUS.glob("*.md")),
               reverse=True)[:3]
print("*.md 기준 상위 3편", truth)
print("코퍼스 디렉터리에 있는 것", sorted(p.name for p in CORPUS.iterdir())[:5], "...")
print("모델의 답이 *.md 상위 3편과 같은가:", all(t in (answer or "") for _, t in truth))

*.md 기준 상위 3편 [(138028, '이승만'), (98758, '서재필'), (91490, '김구')]
코퍼스 디렉터리에 있는 것 ['.gitignore', '3·1_운동.md', 'SOURCES.tsv', 'fetch_wiki_cluster.py', '강우규.md'] ...
모델의 답이 *.md 상위 3편과 같은가: True


### 범위 결정과 격리

코퍼스 디렉터리에는 문서 외에 수집 스크립트와 캐시도 있다. 전용 도구는 `*.md`만 보도록 강제할 수 있지만 코드 실행 도구에서는 무엇을 문서로 셀지 모델이 결정한다.

범위는 지시문에 명시하거나(단, 지켜진다구 확정할 수는 없음) 컨테이너 마운트(비용과 공수가 들지만 좀 더 안전)로 제한할 수 있다. 지시문은 위반될 수 있지만 마운트 경계는 강제된다. 결과가 정답과 같아도 범위 선택이 보장된 것은 아니다.

### 전용 도구와 코드 실행

| | 만든 도구 | 코드 실행 |
|---|---|---|
| 할 수 있는 일 | 우리가 정한 것만 | 파이썬으로 되는 전부 |
| 인자 검증 | 도구 안에서 한다 | 할 수 없다. 경계는 프로세스와 컨테이너뿐 |
| 오류 | 우리가 문구를 쓴다 | 파이썬 트레이스백이 그대로 간다 |
| 검산 | 도구가 맞으면 맞다 | 코드가 남으므로 사람이 다시 돌릴 수 있다 |

코드 실행은 도구 선택 문제를 줄이는 대신 Policy의 경계를 **도구 이름 검사에서 실행 환경 격리**로 옮긴다.
AI 코딩에이전트도 기본도구 외에는 쉘을 사용해서 동작

### 비용 확인

In [19]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 4회
입력 5,003 토큰   출력 866 토큰
누적 $0.0255
